# IND2622 — Experimento II (Q2): versão final

**Q2.** Uma representação semântica mantém potência, duração, janelas e dependências ao preparar tarefas industriais para simulação?

Três rotas partem das mesmas instâncias do benchmark KIT (Ludwig et al., 2019):

- **Rota direta**: o JSON vira duas tabelas (blocos e precedências) e alimenta o modelo de programação. Aqui, o modelo é o problema temporal simples (início de cada bloco, janelas e defasagens), resolvido por Bellman-Ford. A rota só "percebe" um problema se o código quebrar ou se o modelo ficar sem solução.
- **Rota direta com regras em Python** (controle, novo nesta etapa): a mesma rota direta, mas com as mesmas 9 regras da rota semântica programadas em Python e conferidas antes do modelo. Separa o ganho de *ter as regras* do ganho de *declará-las num vocabulário padrão*.
- **Rota semântica**: o JSON vira um grafo RDF. Cada bloco é um `s4ener:Slot`, cada tarefa-base é uma `s4ener:PowerSequence`, e as precedências com defasagem ficam numa extensão local. As restrições são declaradas em SHACL e validadas antes da simulação.

Avaliação: (1) preservação dos campos na ida e volta; (2) viabilidade da programação de referência reconstruída; (3) detecção e localização de incoerências injetadas; (4) falsos alarmes nas instâncias originais; (5) custo de tempo e de código; (6) validação das 1.764 instâncias da base.

O código das rotas está em `kit_rotas.py` e `regras_python.py`, na mesma pasta. A base deve estar descompactada em `data/dataset/archive/`, como no `data.zip`.

In [1]:
import json, time, warnings
from collections import Counter, defaultdict
from multiprocessing import Pool
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import mlflow

import kit_rotas as kr
import regras_python as rp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
mlflow.set_tracking_uri("sqlite:///mlflow.db")

BASE = Path("data/dataset/archive")
SAIDAS = Path("saidas_q2"); SAIDAS.mkdir(exist_ok=True)
SEED = 2026


def mapear(f, itens):
    # paralelo só onde o Python usa fork (Linux); no macOS e no Windows roda em sequência
    import multiprocessing
    if multiprocessing.get_start_method() == "fork":
        with Pool(2) as p:
            return p.map(f, itens, chunksize=4)
    return list(map(f, itens))

2026/11/22 16:51:39 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /usr/local/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## 1. Dados e checagens

Todas as 1.764 instâncias e as 1.764 soluções de referência são lidas para a auditoria. O experimento com as duas rotas usa depois uma amostra.

In [2]:
arquivos = sorted(BASE.glob("instances/*.json"), key=lambda p: int(p.stem.split("_")[1]))
resultados = pd.read_csv(BASE / "results/results.csv")
sol_por_inst = {}
for f in (BASE / "results/solutions").glob("*.json"):
    s = json.load(open(f))
    sol_por_inst[s["instance"]] = f

linhas, lags, extras = [], Counter(), Counter()
for f in arquivos:
    d = json.load(open(f))
    a, J = d["additional"], d["jobs"]
    horizonte = a["generator__days"] * a["generator__timesteps_per_day"]
    seqs = Counter(j["additional"]["superjob_id"] for j in J)
    for j in J:
        for s, v in j["successors"].items():
            lags["negativa" if v["lag"] < 0 else ("zero" if v["lag"] == 0 else "positiva")] += 1
            extras["drain_factor != 0"] += v.get("drain_factor", 0) != 0
            extras["max_recharge != 0"] += v.get("max_recharge", 0) != 0
    linhas.append({
        "arquivo": f.name, "instancia": d["id"], "grupo": a["group_id"], "blocos": a["generator__block_count"],
        "n_blocos": len(J), "n_sequencias": len(seqs), "n_precedencias": sum(len(j["successors"]) for j in J),
        "n_recursos": len(d["resources"]),
        "janela_inviavel": sum(j["release"] + j["duration"] > j["deadline"] for j in J),
        "fora_horizonte": sum(j["release"] < 0 or j["deadline"] > horizonte for j in J),
        "duracao_nao_positiva": sum(j["duration"] <= 0 for j in J),
        "potencia_negativa": sum(v < 0 for j in J for v in j["usages"].values()),
        "tem_solucao": d["id"] in sol_por_inst,
    })
inv = pd.DataFrame(linhas)
print("instâncias:", len(inv), "| grupos:", inv.grupo.nunique(), "| soluções:", inv.tem_solucao.sum())
print("variantes por grupo:", inv.groupby("grupo").size().value_counts().to_dict())
print("recursos por instância:", inv.n_recursos.value_counts().to_dict())
print("defasagens:", dict(lags), "|", dict(extras))
print("problemas de qualidade:", inv[["janela_inviavel", "fora_horizonte", "duracao_nao_positiva",
                                       "potencia_negativa"]].sum().to_dict())
inv.groupby("blocos")[["n_blocos", "n_sequencias", "n_precedencias"]].median().astype(int)

instâncias: 1764 | grupos: 252 | soluções: 1764
variantes por grupo: {7: 252}
recursos por instância: {1: 1764}
defasagens: {'positiva': 2205000, 'negativa': 529200} | {'drain_factor != 0': 0, 'max_recharge != 0': 0}
problemas de qualidade: {'janela_inviavel': 0, 'fora_horizonte': 0, 'duracao_nao_positiva': 0, 'potencia_negativa': 0}


,n_blocos,n_sequencias,n_precedencias
blocos,,,
1,350,350,0
2,700,350,700
3,1050,350,1050
4,1400,350,1400
5,1750,350,1750
7,2450,350,2450
10,3500,350,3500


**Achado da auditoria: a contiguidade dos blocos está implícita.** Nas variantes com mais de um bloco, os blocos de uma tarefa-base formam uma cadeia (cada bloco só começa *duração* passos depois do anterior) e o último aponta de volta para o primeiro com defasagem **negativa** igual à soma das durações anteriores. O ciclo, com soma de defasagens zero, obriga os blocos a rodarem colados, sem intervalo. Isso não está escrito em nenhum campo; está codificado no sinal de um número.

In [3]:
exemplo = json.load(open(BASE / "instances/instance_10.json"))
seq = defaultdict(list)
for j in exemplo["jobs"]:
    seq[j["additional"]["superjob_id"]].append(j)
pd.DataFrame([{"bloco": j["id"], "início mais cedo": j["release"], "fim mais tarde": j["deadline"],
               "duração": j["duration"], "potência": j["usages"]["0"],
               "sucessor (defasagem)": {k: v["lag"] for k, v in j["successors"].items()}}
              for j in seq[6]])

,bloco,início mais cedo,fim mais tarde,duração,potência,sucessor (defasagem)
0,15,302,394,13,76.98,{'16': 13}
1,16,302,394,13,103.28,{'17': 13}
2,17,302,394,13,85.68,{'15': -26}


In [4]:
def checar_solucao_direta(f):
    inst = json.load(open(f))
    sol = json.load(open(sol_por_inst[inst["id"]]))
    inicio = {j["id"]: j["start_time"] for j in sol["jobs"]}
    t = time.perf_counter()
    situacao, _ = kr.rota_direta(inst)
    tempo = time.perf_counter() - t
    return {"arquivo": f.name, "direta_original": situacao, "direta_tempo_s": tempo,
            "direta_solucao_viavel": kr.solucao_viavel_direta(inst, inicio)}

todas = pd.DataFrame(mapear(checar_solucao_direta, arquivos))
inv = inv.merge(todas, on="arquivo")
print("rota direta nas 1.764 originais:", inv.direta_original.value_counts().to_dict())
print("programação de referência viável:", inv.direta_solucao_viavel.mean())

rota direta nas 1.764 originais: {'aceita': 1762, 'inviavel': 2}
programação de referência viável: 0.9988662131519275


**Duas instâncias originais são inviáveis.** A rota direta acusa modelo sem solução em 2 das 1.764 instâncias, e a programação de referência publicada para elas viola as restrições. O arquivo de resultados do benchmark confirma: nelas, a melhor solução aparece com o valor sentinela 1,8 × 10³⁰⁸ e o gap é de 100%. A rota semântica aponta a causa e o lugar.

In [5]:
anomalas = inv[inv.direta_original != "aceita"]
res_idx = resultados.set_index("Instance ID")
diag = []
for r in anomalas.itertuples():
    inst = json.load(open(BASE / "instances" / r.arquivo))
    v = kr.violacoes(kr.para_rdf(inst))
    for regra, foco in v:
        sj = int(foco.split("_seq_")[1])
        b = [j for j in inst["jobs"] if j["additional"]["superjob_id"] == sj]
        ini, fim = min(j["release"] for j in b), max(j["deadline"] for j in b)
        diag.append({"arquivo": r.arquivo, "blocos": r.blocos, "regra": regra, "sequência": sj,
                     "janela (passos)": fim - ini, "soma das durações": sum(j["duration"] for j in b),
                     "durações dos blocos": [j["duration"] for j in b],
                     "melhor solução publicada": res_idx.loc[inst["id"], "Best Found Solution"],
                     "gap publicado": res_idx.loc[inst["id"], "MIP Gap"]})
print("valores sentinela em results.csv:", int((resultados["Best Found Solution"] > 1e300).sum()))
pd.DataFrame(diag)

valores sentinela em results.csv: 2


,arquivo,blocos,regra,sequência,janela (passos),soma das durações,durações dos blocos,melhor solução publicada,gap publicado
0,instance_70.json,10,R7,313,7,10,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]",1.797693e+308,1.0
1,instance_385.json,10,R7,89,18,20,"[2, 2, 2, 2, 2, 2, 2, 2, 2, 2]",1.797693e+308,1.0


## 2. Plano do experimento

| Elemento | Definição |
|---|---|
| Amostra | 10 grupos sorteados × 7 variantes = 70 instâncias (todos os níveis de decomposição) |
| Rotas | direta (tabelas → modelo temporal); direta com as 9 regras em Python; semântica (RDF → SHACL) |
| Erros injetados | 9 tipos, um por cópia, no bloco ou sequência sorteados (seed fixa) |
| Métricas | preservação na ida e volta; viabilidade da solução de referência; detecção; localização; falsos alarmes; tempo |

Resultado de cada rota diante de uma instância:

- **diagnosticado**: rejeitada antes da simulação, com a regra violada e o bloco ou sequência apontados;
- **inviável sem diagnóstico**: o modelo não tem solução, mas não diz por quê nem onde;
- **falha de execução**: o código quebra (erro de tipo, chave inexistente);
- **não detectado**: a instância segue para a simulação como se estivesse correta.

| Erro | Alteração | Regra que deveria pegar (SHACL e Python) |
|---|---|---|
| E1 | fim da janela < início + duração | R4 |
| E2 | duração zero | R1 |
| E3 | potência negativa | R2 |
| E4 | precedência para bloco inexistente | R5 |
| E5 | fim da janela ausente | R3 |
| E6 | duração como texto | R1 |
| E7 | janela da sequência menor que a soma das durações (cada bloco ainda cabe sozinho) | R7 |
| E8 | defasagem de retorno −5: permite intervalos entre blocos (contiguidade perdida, modelo ainda viável) | R8 |
| E9 | defasagem de retorno +5: ciclo impossível | R8 |

In [6]:
rng = np.random.default_rng(SEED)
grupos = sorted(rng.choice(sorted(inv.grupo.unique()), 10, replace=False))
amostra = inv[inv.grupo.isin(grupos)].sort_values(["grupo", "blocos"]).reset_index(drop=True)
print(len(amostra), "instâncias | blocos por instância:", amostra.n_blocos.min(), "a", amostra.n_blocos.max())
amostra.groupby("blocos").n_blocos.describe()[["min", "50%", "max"]].astype(int)

70 instâncias | blocos por instância: 215 a 4850


,min,50%,max
blocos,,,
1,215,357,485
2,430,715,970
3,645,1072,1455
4,860,1430,1940
5,1075,1787,2425
7,1505,2502,3395
10,2150,3575,4850


## 3. Execução

### 3.1 Instâncias originais: preservação, falsos alarmes e solução de referência

In [7]:
def avaliar_original(arquivo):
    inst = json.load(open(BASE / "instances" / arquivo))
    sol = json.load(open(sol_por_inst[inst["id"]]))
    inicio = {j["id"]: j["start_time"] for j in sol["jobs"]}
    ref = [{"id": j["id"], "release": j["release"], "deadline": j["deadline"], "duration": j["duration"],
            "usages": {"0": float(j["usages"]["0"])}, "superjob_id": j["additional"]["superjob_id"],
            "successors": {k: {"lag": v["lag"]} for k, v in j["successors"].items()}} for j in inst["jobs"]]

    def comparar(rec):
        campos = Counter()
        porj = {j["id"]: j for j in rec}
        for r in ref:
            o = porj.get(r["id"], {})
            campos["potência"] += o.get("usages") == r["usages"]
            campos["duração"] += o.get("duration") == r["duration"]
            campos["janela"] += (o.get("release"), o.get("deadline")) == (r["release"], r["deadline"])
            campos["dependências"] += o.get("successors") == r["successors"]
            campos["sequência"] += o.get("superjob_id") == r["superjob_id"]
        return {k: v / len(ref) for k, v in campos.items()}

    t = time.perf_counter(); tabelas = kr.rota_direta_tabelas(inst); situ_d, _ = kr.rota_direta(inst)
    t_d = time.perf_counter() - t
    t = time.perf_counter(); v_py = rp.violacoes_python(inst); t_py = time.perf_counter() - t
    t = time.perf_counter(); g = kr.para_rdf(inst); t_rdf = time.perf_counter() - t
    t = time.perf_counter(); v = kr.violacoes(g); t_shacl = time.perf_counter() - t
    g_sol = kr.para_rdf(inst, inicio)
    v_sol = kr.violacoes(g_sol, kr._shapes_sol)
    return {"arquivo": arquivo, "n_triplas": len(g), "direta": situ_d, "semantica_violacoes": len(v),
            "python_violacoes": len(v_py), "tempo_python_s": t_py,
            "direta_preserva": comparar(kr.direta_para_json(*tabelas)),
            "semantica_preserva": comparar(kr.rdf_para_json(g)),
            "semantica_solucao_viavel": len(v_sol) == 0,
            "tempo_direta_s": t_d, "tempo_rdf_s": t_rdf, "tempo_shacl_s": t_shacl}

orig = pd.DataFrame(mapear(avaliar_original, list(amostra.arquivo)))
orig = amostra[["arquivo", "blocos", "n_blocos", "direta_solucao_viavel"]].merge(orig, on="arquivo")

preserva = pd.DataFrame({
    "rota direta": pd.DataFrame(orig.direta_preserva.tolist()).mean(),
    "rota semântica": pd.DataFrame(orig.semantica_preserva.tolist()).mean(),
}).mul(100).round(2)
preserva.index.name = "% dos blocos com o campo idêntico após ida e volta"
print("falsos alarmes — direta:", (orig.direta != "aceita").sum(), "| direta + Python:", (orig.python_violacoes > 0).sum(),
      "| semântica:", (orig.semantica_violacoes > 0).sum())
print("solução de referência viável — direta:", orig.direta_solucao_viavel.mean(),
      "| semântica:", orig.semantica_solucao_viavel.mean())
preserva

falsos alarmes — direta: 0 | direta + Python: 0 | semântica: 0
solução de referência viável — direta: 1.0 | semântica: 1.0


,rota direta,rota semântica
% dos blocos com o campo idêntico após ida e volta,,
potência,100.0,100.0
duração,100.0,100.0
janela,100.0,100.0
dependências,100.0,100.0
sequência,100.0,100.0


### 3.2 Injeção de erros

In [8]:
def avaliar_erro(args):
    arquivo, tipo, seed = args
    inst = json.load(open(BASE / "instances" / arquivo))
    x, alvo = kr.injetar(inst, tipo, np.random.default_rng(seed))
    if x is None:
        return None
    situ_d, det_d = kr.rota_direta(x)
    situ_p, regras_p, viol_p = rp.rota_direta_validada(x)
    situ_s, regras, viol = kr.rota_semantica(x)
    seq_alvo = next(j["additional"]["superjob_id"] for j in x["jobs"] if j["id"] == alvo)

    def localiza(v):
        return any(f.endswith(f"_bloco_{alvo}") or f"_bloco_{alvo}_prec_" in f or f.endswith(f"_seq_{seq_alvo}")
                   for _, f in v)

    return {"arquivo": arquivo, "tipo": tipo, "direta": situ_d, "direta_detalhe": det_d,
            "python": situ_p, "regras_python": ",".join(regras_p or []), "localizou_python": localiza(viol_p),
            "semantica": situ_s, "regras": ",".join(regras or []), "n_violacoes": len(viol),
            "localizou": localiza(viol),
            "mesmas_violacoes": sorted({(r, f.split("#")[-1]) for r, f in viol}) == sorted(set(viol_p))}

tarefas = [(a, t, SEED + 1000 * i + k) for i, a in enumerate(amostra.arquivo) for k, t in enumerate(kr.TIPOS)]
t0 = time.time()
inj = pd.DataFrame([r for r in mapear(avaliar_erro, tarefas) if r is not None])
print(len(inj), "instâncias alteradas em", round((time.time() - t0) / 60, 1), "min")
inj.to_csv(SAIDAS / "injecao_erros.csv", index=False)

600 instâncias alteradas em 19.7 min


## 4. Resultados

In [9]:
def classe_direta(s):
    return {"aceita": "não detectado", "inviavel": "inviável sem diagnóstico", "falha": "falha de execução"}[s]

def classe_regras(situacao, localizou):
    if situacao == "rejeitada":
        return "diagnosticado" if localizou else "rejeitado sem localizar"
    return "não detectado" if situacao == "aceita" else "falha de execução"

inj["resultado_direta"] = inj.direta.map(classe_direta)
inj["resultado_python"] = [classe_regras(a, b) for a, b in zip(inj.python, inj.localizou_python)]
inj["resultado_semantica"] = [classe_regras(a, b) for a, b in zip(inj.semantica, inj.localizou)]
print("cópias em que Python e SHACL apontam exatamente as mesmas violações:", f"{inj.mesmas_violacoes.mean():.1%}")

ORDEM = ["diagnosticado", "rejeitado sem localizar", "inviável sem diagnóstico", "falha de execução", "não detectado"]
tab = (pd.concat([inj.groupby("tipo").resultado_direta.value_counts(normalize=True).rename("direta"),
                  inj.groupby("tipo").resultado_semantica.value_counts(normalize=True).rename("semântica")], axis=1)
       .fillna(0).mul(100).round(1))
resumo_tipo = pd.DataFrame({
    "n": inj.groupby("tipo").size(),
    "direta: detecta (%)": inj.groupby("tipo").direta.apply(lambda s: 100 * (s != "aceita").mean()).round(1),
    "direta: resultado típico": inj.groupby("tipo").resultado_direta.agg(lambda s: s.value_counts().index[0]),
    "Python: detecta (%)": inj.groupby("tipo").python.apply(lambda s: 100 * (s == "rejeitada").mean()).round(1),
    "Python: localiza (%)": inj.groupby("tipo").localizou_python.mean().mul(100).round(1),
    "semântica: detecta (%)": inj.groupby("tipo").semantica.apply(lambda s: 100 * (s == "rejeitada").mean()).round(1),
    "semântica: localiza (%)": inj.groupby("tipo").localizou.mean().mul(100).round(1),
    "regras disparadas": inj.groupby("tipo").regras.agg(lambda s: ", ".join(sorted(set(",".join(s).split(",")) - {""}))),
})
resumo_tipo.to_csv(SAIDAS / "resumo_por_tipo.csv")
resumo_tipo

cópias em que Python e SHACL apontam exatamente as mesmas violações: 100.0%


,n,direta: detecta (%),direta: resultado típico,Python: detecta (%),Python: localiza (%),semântica: detecta (%),semântica: localiza (%),regras disparadas
tipo,,,,,,,,
E1_janela_menor_que_duracao,70,100.0,inviável sem diagnóstico,100.0,100.0,100.0,100.0,"R4, R7, R9"
E2_duracao_zero,70,0.0,não detectado,100.0,100.0,100.0,100.0,R1
E3_potencia_negativa,70,0.0,não detectado,100.0,100.0,100.0,100.0,R2
E4_precedencia_inexistente,70,100.0,falha de execução,100.0,100.0,100.0,100.0,R5
E5_campo_ausente,70,0.0,não detectado,100.0,100.0,100.0,100.0,R3
E6_tipo_invalido,70,100.0,falha de execução,100.0,100.0,100.0,100.0,R1
E7_janela_da_sequencia,60,100.0,inviável sem diagnóstico,100.0,100.0,100.0,100.0,"R7, R9"
E8_contiguidade_quebrada,60,0.0,não detectado,100.0,100.0,100.0,100.0,R8
E9_ciclo_inviavel,60,100.0,inviável sem diagnóstico,100.0,100.0,100.0,100.0,R8


In [10]:
geral = pd.DataFrame({
    "rota direta": inj.resultado_direta.value_counts(normalize=True),
    "direta + regras em Python": inj.resultado_python.value_counts(normalize=True),
    "rota semântica": inj.resultado_semantica.value_counts(normalize=True),
}).reindex(ORDEM).fillna(0).mul(100).round(1)
geral.loc["detecção total"] = 100 - geral.loc["não detectado"]
geral.to_csv(SAIDAS / "resumo_geral.csv")
geral

,rota direta,direta + regras em Python,rota semântica
diagnosticado,0.0,100.0,100.0
rejeitado sem localizar,0.0,0.0,0.0
inviável sem diagnóstico,31.7,0.0,0.0
falha de execução,23.3,0.0,0.0
não detectado,45.0,0.0,0.0
detecção total,55.0,100.0,100.0


In [11]:
cores = {"diagnosticado": "#167C7A", "rejeitado sem localizar": "#59C4BB", "inviável sem diagnóstico": "#235D9F",
         "falha de execução": "#B5651D", "não detectado": "#C9D3DD"}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
for ax, col, nome in zip(axes, ["resultado_direta", "resultado_python", "resultado_semantica"],
                         ["Rota direta", "Direta + regras em Python", "Rota semântica"]):
    m = pd.crosstab(inj.tipo, inj[col], normalize="index").reindex(columns=ORDEM).fillna(0) * 100
    esq = np.zeros(len(m))
    for c in ORDEM:
        ax.barh(m.index, m[c], left=esq, color=cores[c], label=c)
        esq += m[c].to_numpy()
    ax.set_title(nome); ax.set_xlim(0, 100); ax.set_xlabel("% das instâncias alteradas")
axes[0].invert_yaxis()
axes[1].legend(loc="lower center", bbox_to_anchor=(0.5, -0.35), ncol=5, fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(SAIDAS / "deteccao_por_tipo.png", dpi=150, bbox_inches="tight")
plt.close(fig)

In [12]:
tempo = orig.groupby("blocos").agg(n_blocos=("n_blocos", "median"), triplas=("n_triplas", "median"),
                                   direta_s=("tempo_direta_s", "median"), python_s=("tempo_python_s", "median"),
                                   rdf_s=("tempo_rdf_s", "median"),
                                   shacl_s=("tempo_shacl_s", "median")).round(3)
tempo.to_csv(SAIDAS / "tempo_por_decomposicao.csv")
orig.drop(columns=["direta_preserva", "semantica_preserva"]).to_csv(SAIDAS / "originais.csv", index=False)
preserva.to_csv(SAIDAS / "preservacao.csv")
tempo

,n_blocos,triplas,direta_s,python_s,rdf_s,shacl_s
blocos,,,,,,
1,357.5,3933.5,0.005,0.001,0.043,0.621
2,715.0,10368.5,0.009,0.004,0.148,1.797
3,1072.5,15016.0,0.010,0.005,0.180,2.221
4,1430.0,19663.5,0.013,0.006,0.282,2.905
5,1787.5,24311.0,0.015,0.008,0.529,3.672
7,2502.5,33606.0,0.020,0.011,0.669,4.860
10,3575.0,47548.5,0.028,0.016,1.030,6.779


### 4.1 Custo de escrever as regras

As duas implementações das regras chegam ao mesmo resultado. A diferença está na forma: em SHACL as regras são dados declarados à parte do programa, com nome e mensagem, e podem ser lidas e reutilizadas por outra ferramenta que entenda o padrão; em Python elas são código, presas a este formato de JSON.

In [13]:
import inspect

def linhas_uteis(texto):
    return sum(1 for l in texto.splitlines() if l.strip() and not l.strip().startswith(("#", chr(34) * 3)))

esforco = pd.DataFrame([
    {"implementação": "SHACL (regras R0–R9)", "linhas": linhas_uteis(kr.SHAPES),
     "onde vivem as regras": "arquivo de regras, separado do código", "mensagem por regra": "sim, declarada na regra"},
    {"implementação": "Python (regras R0–R9)", "linhas": linhas_uteis(inspect.getsource(rp.violacoes_python)),
     "onde vivem as regras": "dentro do código", "mensagem por regra": "só o código da regra"},
    {"implementação": "Conversão JSON → RDF (só na semântica)", "linhas": linhas_uteis(inspect.getsource(kr.para_rdf)),
     "onde vivem as regras": "—", "mensagem por regra": "—"},
])
esforco.to_csv(SAIDAS / "esforco_implementacao.csv", index=False)
esforco

,implementação,linhas,onde vivem as regras,mensagem por regra
0,SHACL (regras R0–R9),49,"arquivo de regras, separado do código","sim, declarada na regra"
1,Python (regras R0–R9),52,dentro do código,só o código da regra
2,Conversão JSON → RDF (só na semântica),34,—,—


### 4.2 As 1.764 instâncias da base

As duas implementações das regras rodam em todas as instâncias originais. Uma instância que viola alguma regra é um problema real da base publicada.

In [14]:
def validar_base(f):
    inst = json.load(open(f))
    t = time.perf_counter(); vp = rp.violacoes_python(inst); tp = time.perf_counter() - t
    t = time.perf_counter(); vs = kr.violacoes(kr.para_rdf(inst)); ts = time.perf_counter() - t
    return {"arquivo": f.name, "python": ",".join(sorted({r for r, _ in vp})), "semantica": ",".join(sorted({r for r, _ in vs})),
            "n_python": len(vp), "n_semantica": len(vs), "tempo_python_s": tp, "tempo_semantica_s": ts}

t0 = time.time()
base = pd.DataFrame(mapear(validar_base, arquivos)).merge(inv[["arquivo", "blocos", "n_blocos", "direta_original"]], on="arquivo")
base.to_csv(SAIDAS / "validacao_base_completa.csv", index=False)
print(len(base), "instâncias em", round((time.time() - t0) / 60, 1), "min")
print("com violação — Python:", (base.n_python > 0).sum(), "| SHACL:", (base.n_semantica > 0).sum(),
      "| rota direta inviável:", (base.direta_original != "aceita").sum())
base[(base.n_python > 0) | (base.n_semantica > 0)]

1764 instâncias em 53.1 min
com violação — Python: 2 | SHACL: 2 | rota direta inviável: 2


,arquivo,python,semantica,n_python,n_semantica,tempo_python_s,tempo_semantica_s,blocos,n_blocos,direta_original
69,instance_70.json,R7,R7,1,1,0.015864,7.454151,10,3350,inviavel
384,instance_385.json,R7,R7,1,1,0.014911,8.243055,10,3800,inviavel


In [15]:
custo = base.groupby("blocos").agg(n_blocos=("n_blocos", "median"), python_s=("tempo_python_s", "median"),
                                   semantica_s=("tempo_semantica_s", "median")).round(3)
custo.to_csv(SAIDAS / "tempo_base_completa.csv")
print("tempo total — Python:", round(base.tempo_python_s.sum() / 60, 1), "min | SHACL:", round(base.tempo_semantica_s.sum() / 60, 1), "min")
custo

tempo total — Python: 0.2 min | SHACL: 105.5 min


,n_blocos,python_s,semantica_s
blocos,,,
1,350.0,0.001,0.648
2,700.0,0.004,1.735
3,1050.0,0.005,2.551
4,1400.0,0.007,3.178
5,1750.0,0.008,3.895
7,2450.0,0.011,5.287
10,3500.0,0.016,7.474


In [16]:
mlflow.set_experiment("IND2622_Q2_final")
COL = {"direta": "resultado_direta", "python": "resultado_python", "semantica": "resultado_semantica"}
FA = {"direta": (orig.direta != "aceita").sum(), "python": (orig.python_violacoes > 0).sum(),
      "semantica": (orig.semantica_violacoes > 0).sum()}
for rota in ["direta", "python", "semantica"]:
    with mlflow.start_run(run_name=f"Q2|{rota}"):
        mlflow.set_tags({"etapa": "3", "questao": "Q2"})
        mlflow.log_params({"rota": rota, "n_instancias": len(amostra), "grupos": len(grupos), "seed": SEED,
                           "tipos_erro": len(kr.TIPOS)})
        col = COL[rota]
        mlflow.log_metric("deteccao_total", float((inj[col] != "não detectado").mean()))
        mlflow.log_metric("diagnostico_localizado", float((inj[col] == "diagnosticado").mean()))
        mlflow.log_metric("falsos_alarmes", float(FA[rota]))
        nome_pres = "rota semântica" if rota == "semantica" else "rota direta"
        for campo, val in preserva[nome_pres].items():
            mlflow.log_metric(f"preserva_{campo}", float(val))
        for tipo, sub in inj.groupby("tipo"):
            mlflow.log_metric(f"detecta_{tipo[:2]}", float((sub[col] != "não detectado").mean()))
with mlflow.start_run(run_name="Q2|resumo"):
    mlflow.set_tags({"etapa": "3", "questao": "Q2", "tipo": "resumo"})
    mlflow.log_table(resumo_tipo.reset_index(), "resumo_por_tipo.json")
    mlflow.log_table(geral.reset_index(), "resumo_geral.json")
    mlflow.log_table(esforco, "esforco_implementacao.json")
    mlflow.log_table(base, "validacao_base_completa.json")
    mlflow.log_artifact("regras_python.py")
    mlflow.log_artifact(str(SAIDAS / "deteccao_por_tipo.png"))
    mlflow.log_text(kr.SHAPES, "regras_shacl.ttl")
    mlflow.log_text(kr.ONTOLOGIA, "extensao_local.ttl")

2026/11/22 18:09:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/11/22 18:09:48 INFO mlflow.store.db.utils: Updating database tables


2026/11/22 18:09:50 INFO mlflow.tracking.fluent: Experiment with name 'IND2622_Q2_final' does not exist. Creating a new experiment.
